<a href="https://colab.research.google.com/github/bitsteller/dataanalys-for-kollektivtrafik/blob/main/Kod/Laboration%203/Uppgift4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboration 3 – Uppgift 4

I den här uppgiften ska vi använda maskininlärning för att prediktera upphållstiden för ett tåg i Hässleholm.

## Jobba så här
1. Kör exempelcellen i varje avsnitt.
2. Fyll i övningscellen under.
3. Tolka resultatet innan du går vidare.


In [ ]:
# Kör denna cell först
import polars as pl
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error


BASE_URL = "https://raw.githubusercontent.com/bitsteller/dataanalys-for-kollektivtrafik/main/Data/Laboration%203"
trains = pl.read_parquet(f"{BASE_URL}/trains.parquet")

print("Totalt antal rader:", trains.height)

## 1) Skapa tränings- och testdata

Vi delar datan i tränings- och testdata för att kunna utvärdera modellen på data den inte sett.

- `sample(...)`: gör att raderna kastas om i slumpmässig ordning
- `slice(...)`: plocka ut vissa rader av tabellen, ungefär som slices för listor (se uppgift 1)


In [ ]:
#slumpar om data och delar i 80/20
shuffled = trains.sample(fraction=1.0, shuffle=True, seed=42)
cutoff = int(shuffled.height * 0.8)

training = shuffled.slice(0, cutoff)
test = shuffled.slice(cutoff)

print("Train:", training.height)
print("Test:", test.height)


In [ ]:
# Övning: Ändra koden så att vi får en 70/30-split i stället och skriv ut antal rader.
shuffled = trains.sample(fraction=1.0, shuffle=True, seed=42)
cutoff_7030 = int(shuffled.height * 0.8)
train_7030 = shuffled.slice(0, cutoff_7030)
test_7030 = shuffled.slice(cutoff_7030)

print("Train 70%:", train_7030.height)
print("Test 30%:", test_7030.height)


## 2) Enkel linjär modell

Vi börjar med en enkel linjär modell. Vi ska modellera `ActualDwellTime` som funktion av `PlannedDwellTime`, vilket skrivs som
`ActualDwellTime ~ PlannedDwellTime`. OLS står för *ordinary least squares*.

- `sms.ols()`: skapar en ny linjär regressionsmodell
- `fit()`: tränar modellen.
- `summary()`: visar koefficienter, p-värden och modellmått.

In [ ]:
model1 = smf.ols("ActualDwellTime ~ PlannedDwellTime", data=training)
model1 = model1.fit()

In [ ]:
# Övning: Använd summary() för att skriva ut information om modellen.



Titta på resultaten och svara på frågorna:

1. Är sambandet mellan PlannedDwellTime och ActualDwellTime signifikant?
2. Hur mycket ökar den faktiska uppehållstiden när den planerade ökar med 1 min?
3. Hur bra är modellen på att förklara variansen i träningsdatat? (titta på R^2)?

Tips: Titta på föreläsningsbilderna för att se vilka värden som är relevanta att titta på.

# 3) Utvärdera modellen med testdata

För att kunna bedöma om modellen om modellen inte drabbas av overfitting (är för flexibel), utvärderar vi modellen genom att göra prognoser på testdatat som modellen inte har sett än. Genomsnittligt fel kallas även för *mean average error (MAE)*.


In [ ]:
# Beräkna training error som genomsnittlig skillnad mellan faktiskt och predikterat värde
training_error = (training["ActualDwellTime"] - pl.Series(model1.predict(training))).abs().mean()
print("Training MAE:",training_error)

In [ ]:
#Övning: Beräkna nu test error för modellen genom att göra samma beräkning fast för testdata
#Om test error är mycket högre än training error, drabbas modellen av overfitting.


## 3) Linjär regression med flera förklaringsvariabler

Nu modellerar vi `ActualDwellTime` som funktion av både `PlannedDwellTime` och `ToLocationSignature`.


In [ ]:
model2 = smf.ols("ActualDwellTime ~ PlannedDwellTime + TrackAtLocation", data=training.to_pandas())
model2 = model2.fit()

model2.summary()


In [ ]:
#Beräkna test error för den nya modellen
test_error = (test["ActualDwellTime"] - pl.Series(model2.predict(test.to_pandas()))).abs().mean()
print("Test MAE:", test_error)


Är det nya modellen bättre än den enkla modellen med en förklaringsvariabel?

Använd resultaten och test error för bedömningen.

In [ ]:
#Övning: Gör nu en egen modell genom att lägga till fler förklaringsvariabler.
#Försök att hitta en modell med högre R^2 och lägre test error än modell2.

model3 = smf.ols("ActualDwellTime ~ 1", data=training.to_pandas()) #Byt 1 till förklaringsvariabeler
model3 = model3.fit()

model3.summary()

In [ ]:
#Beräkna test error för den tredje modellen
test_error = (test["ActualDwellTime"] - pl.Series(model3.predict(test.to_pandas()))).abs().mean()
print("Test MAE:", test_error)

## Bonus) Beslutsträd

Istället för att använda linjär regression kan vi använda andra typer av modeller. En metod är till exempel beslutsträd. Här tränar vi ett beslutsträd för att prediktera `ActualDwellTime` med förklaringsvariablerna `PlannedDwellTime` och `TrackAtLocation`.

- `to_dummies(...)`: kodar kategoriska variabler med dummyvariabler (en variabel för varje kategori som är sant/falsk)
- `DecisionTreeRegressor(...)`: tränar en trädmodell.
- `plot_tree`: skapar ett träddiagram

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree

# Förklaringsvariabler
bonus_numerical_features = ["PlannedDwellTime"]
bonus_categorical_features = ["ToLocationSignature", "ProductName"]
bonus_features = bonus_numerical_features + bonus_categorical_features


#Skapa träningsdata
bonus_train = training.select(bonus_features + ["ActualDwellTime"]).drop_nulls()
X_train_bonus = (
    bonus_train
    .select(bonus_features)
    .to_dummies(columns=bonus_categorical_features)
    .to_pandas()
)
y_train_bonus = bonus_train["ActualDwellTime"].to_pandas()

#Träna modellen
tree = DecisionTreeRegressor(max_depth=4)
tree.fit(X_train_bonus, y_train_bonus)

#Plotta trädmodellen
plt.figure(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=X_train_bonus.columns,
    label="root",
    impurity=False,
    precision=2,
    proportion=True,
    max_depth=4,
    fontsize=8,
)
plt.show()

#Beräkna test error
bonus_test = test.select(bonus_features + ["ActualDwellTime"]).drop_nulls()
X_test_bonus = (
    bonus_test
    .select(bonus_features)
    .to_dummies(columns=bonus_categorical_features)
    .to_pandas()
)
X_test_bonus = X_test_bonus.reindex(columns=X_train_bonus.columns, fill_value=0)
y_test_bonus = bonus_test["ActualDwellTime"].to_pandas()
pred_tree = tree.predict(X_test_bonus)
print("Test MAE tree:", round(mean_absolute_error(y_test_bonus, pred_tree), 3))


1. Titta på trädet. Vilken upphållstid predikteras för ett Örestundståg med planerat uppehållstid mindre än 4.5 minuter?
2. Testa att minska eller öka `max_depth` för modellen (dvs. minska/öka antalet nivåer i trädet). Blir modellen bättre eller sämre?